In [13]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import (
    LinearRegression,
    DecisionTreeRegressor,
    RandomForestRegressor,
    GBTRegressor,
    GeneralizedLinearRegression
)
from pyspark.ml.evaluation import RegressionEvaluator
import pandas as pd

In [20]:
df = spark.read.csv("processed.csv", header=True, inferSchema=True)

In [21]:
from pyspark.sql.functions import col

cols = [
    "growth_rate",
    "retail_and_recreation_percent_change_from_baseline",
    "workplaces_percent_change_from_baseline",
    "residential_percent_change_from_baseline",
    "Confirmed"
]

for c in cols:
    df = df.withColumn(c, col(c).cast("double"))

In [22]:
from pyspark.ml.feature import VectorAssembler

feature_cols = [
    "growth_rate",
    "retail_and_recreation_percent_change_from_baseline",
    "workplaces_percent_change_from_baseline",
    "residential_percent_change_from_baseline"
]

assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")

model_df = assembler.transform(df).select("features", col("Confirmed").alias("label"))

In [28]:
from pyspark.ml.feature import StandardScaler

scaler = StandardScaler(inputCol="features", outputCol="scaled_features")

scaler_model = scaler.fit(model_df)
model_df = scaler_model.transform(model_df).select("scaled_features", "label")

# rename for compatibility
model_df = model_df.withColumnRenamed("scaled_features", "features")

In [29]:
train_data, test_data = model_df.randomSplit([0.8, 0.2], seed=42)

print("Train rows:", train_data.count())
print("Test rows:", test_data.count())

Train rows: 422456
Test rows: 105714


## Regression from here

In [30]:
from pyspark.ml.regression import (
    LinearRegression,
    DecisionTreeRegressor,
    RandomForestRegressor,
    GBTRegressor,
    GeneralizedLinearRegression
)

from pyspark.ml.evaluation import RegressionEvaluator
import pandas as pd


In [31]:
models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(),
    "Random Forest": RandomForestRegressor(numTrees=50),
    "GBT": GBTRegressor(maxIter=50, maxDepth=5),
    "GLR": GeneralizedLinearRegression()
}

evaluator = RegressionEvaluator(labelCol="label", metricName="rmse")
rmse_eval = RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="rmse")
mae_eval = RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="mae")
r2_eval = RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="r2")

results = []
trained_models = {}

In [32]:
results = []
trained_models = {}

for name, model in models.items():
    print(f"Training {name}...")

    m = model.fit(train_data)
    preds = m.transform(test_data)

    rmse = rmse_eval.evaluate(preds)
    mae = mae_eval.evaluate(preds)
    r2 = r2_eval.evaluate(preds)

    results.append({
        "Model": name,
        "RMSE": rmse,
        "MAE": mae,
        "R2": r2
    })

    trained_models[name] = m

Training Linear Regression...
Training Decision Tree...
Training Random Forest...
Training GBT...
Training GLR...


In [34]:
results_df = pd.DataFrame(results).sort_values(by="RMSE")
results_df

,Model,RMSE,MAE,R2
3,GBT,9.657315e+06,7.177114e+06,0.600624
2,Random Forest,1.034104e+07,8.588348e+06,0.542072
1,Decision Tree,1.052031e+07,8.196267e+06,0.526057
0,Linear Regression,1.172105e+07,9.880568e+06,0.411696
4,GLR,1.172105e+07,9.880568e+06,0.411696


## Classification from here

In [39]:
from pyspark.sql.functions import when, col

df_class = df.withColumn(
    "risk",
    when(col("Confirmed") < 10000, 0)
    .when(col("Confirmed") < 100000, 1)
    .otherwise(2)
)

# 🔥 IMPORTANT: CAST TO INT
df_class = df_class.withColumn("risk", col("risk").cast("int"))
df_class = df_class.dropna(subset=["risk"])

In [40]:
df_class.select("risk").distinct().show()


+----+
|risk|
+----+
|   1|
|   2|
|   0|
+----+



In [42]:
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")

class_df = assembler.transform(df_class).select("features", col("risk").alias("label"))

train_c, test_c = class_df.randomSplit([0.8, 0.2], seed=42)

In [43]:
from pyspark.ml.classification import (
    LogisticRegression,
    DecisionTreeClassifier,
    RandomForestClassifier,
    GBTClassifier
)

In [45]:
class_models = {
    "Logistic Regression": LogisticRegression(),
    "Decision Tree": DecisionTreeClassifier(),
    "Random Forest": RandomForestClassifier(),
}

from pyspark.ml.evaluation import MulticlassClassificationEvaluator

accuracy_eval = MulticlassClassificationEvaluator(metricName="accuracy")
f1_eval = MulticlassClassificationEvaluator(metricName="f1")
precision_eval = MulticlassClassificationEvaluator(metricName="weightedPrecision")
recall_eval = MulticlassClassificationEvaluator(metricName="weightedRecall")

class_results = []

for name, model in class_models.items():
    print(f"Training {name}...")

    m = model.fit(train_c)
    preds = m.transform(test_c)

    acc = accuracy_eval.evaluate(preds)
    f1 = f1_eval.evaluate(preds)
    prec = precision_eval.evaluate(preds)
    rec = recall_eval.evaluate(preds)

    class_results.append({
        "Model": name,
        "Accuracy": acc,
        "F1 Score": f1,
        "Precision": prec,
        "Recall": rec
    })

Training Logistic Regression...
Training Decision Tree...
Training Random Forest...


In [46]:
class_results_df = pd.DataFrame(class_results).sort_values(by="Accuracy", ascending=False)
class_results_df

,Model,Accuracy,F1 Score,Precision,Recall
2,Random Forest,0.915924,0.909656,0.909708,0.915924
1,Decision Tree,0.911923,0.906843,0.905219,0.911923
0,Logistic Regression,0.890015,0.849477,0.830427,0.890015


In [50]:
pdf = df.toPandas()

In [51]:
from sklearn.ensemble import GradientBoostingRegressor
import joblib

X = pdf[[
    "growth_rate",
    "retail_and_recreation_percent_change_from_baseline",
    "workplaces_percent_change_from_baseline",
    "residential_percent_change_from_baseline"
]]

y = pdf["Confirmed"]

model = GradientBoostingRegressor()
model.fit(X, y)

# Save model
import os
os.makedirs("models", exist_ok=True)

joblib.dump(model, "models/best_model.pkl")

print("✅ Model saved successfully in models/best_model.pkl")

✅ Model saved successfully in models/best_model.pkl
